# Finding the Tweets with Negative Sentiment about a Product 

https://www.nlplanet.org/course-practical-nlp/02-practical-nlp-first-tasks/06-tweets-sentiment

In [1]:
from datasets import load_dataset
from transformers import pipeline
import pandas as pd

In [ ]:
# download the tweets dataset (different dataset, the one in the tutorial errors)
dataset = load_dataset("zeroshot/twitter-financial-news-sentiment"
                       ,split="train"
                       ,cache_dir=r"../data/cache"
                       #,streaming=True
                       ,trust_remote_code=True)

In [21]:
dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 9543
})

In [19]:
# convert dataset to pandas dataframe
df = pd.DataFrame(dataset).drop("label", axis=1)
df.head()

,text
0,$BYND - JPMorgan reels in expectations on Beyo...
1,$CCL $RCL - Nomura points to bookings weakness...
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ..."
3,$ESS: BTIG Research cuts to Neutral https://t....
4,$FNKO - Funko slides after Piper Jaffray PT cu...


In [20]:
# download pre-trained tweet sentiment model
model = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest", device=0)

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

c:\Users\TristramArmour\anaconda3\envs\learning\Lib\site-packages\huggingface_hub\file_download.py:147: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\TristramArmour\.cache\huggingface\hub\models--cardiffnlp--twitter-roberta-base-sentiment-latest. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [22]:
# compute the sentiment of each tweet using the model
all_texts = df["text"].values.tolist()
all_sentiments = model(all_texts)
df["sentiment_label"] = [d["label"] for d in all_sentiments]
df["sentiment_score"] = [d["score"] for d in all_sentiments]
df.head()

,text,sentiment_label,sentiment_score
0,$BYND - JPMorgan reels in expectations on Beyo...,neutral,0.763711
1,$CCL $RCL - Nomura points to bookings weakness...,neutral,0.544785
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",neutral,0.495968
3,$ESS: BTIG Research cuts to Neutral https://t....,neutral,0.794447
4,$FNKO - Funko slides after Piper Jaffray PT cu...,neutral,0.893191


In [23]:
negative_tweets = df[df["sentiment_label"] == "negative"]
top_negative_tweets = negative_tweets.sort_values(by="sentiment_score", ascending=False)
top_negative_tweets = top_negative_tweets.reset_index()

print("Negative tweets:")
for i, row in top_negative_tweets.iterrows():
  print(f"- {row['text']} ({row['sentiment_score']:.3f})")
  if i == 10:
    break

Negative tweets:
- Wow this Dallas team sucks #DALvsCHI (0.947)
- This Is the Worst Way to Afford the Holidays (0.943)
- What kind of an animal is this? (0.938)
- Incompetence, not ideology, is the worst thing about this election https://t.co/fOEQ6ytIRQ (0.938)
- #HunterBiden is just killing my life .... it's so hard to be White now in America ..... it's like being William hun… https://t.co/uHk95qx1Ig (0.937)
- 25 Worst Holiday Storms of All Time (0.935)
- This Is the Unhealthiest State in America (0.935)
- NEW YORK POST EDITORIAL. “Worse Than Pointless. All those hours of televised testimony plainly failed on their anno… https://t.co/1XAuOFHyVk (0.934)
- 500 Million Pieces of Trash Cover This State’s Roads (0.931)
- $1 gasoline is back -- and it's a terrible sign for the economy https://t.co/5vCkppVHOV (0.927)
- $1 gasoline is back -- and it's a terrible sign for the economy https://t.co/4WRKl5WZ4N (0.927)
